In [19]:
import pandas as pd
import numpy as np

# 1. Load the data
matches = pd.read_csv('matches.csv')
deliveries = pd.read_csv('deliveries.csv')

# 2. Get first innings total runs for each match
first_inning = deliveries[deliveries['inning'] == 1]
total_score_df = first_inning.groupby('match_id')['total_runs'].sum().reset_index()
# Rename to match the exact column name your Streamlit app expects
total_score_df.rename(columns={'total_runs': 'total_runs_x'}, inplace=True) 

# 3. Merge match info with the target score
match_df = matches.merge(total_score_df, left_on='id', right_on='match_id')

# 4. Filter for second innings and merge back match details
second_inning = deliveries[deliveries['inning'] == 2]
delivery_df = second_inning.merge(match_df[['id', 'city', 'winner', 'total_runs_x']], left_on='match_id', right_on='id')

# 5. Calculate Real-Time Features
delivery_df['current_score'] = delivery_df.groupby('match_id')['total_runs'].cumsum()
# Target is first innings score + 1
delivery_df['runs_left'] = delivery_df['total_runs_x'] - delivery_df['current_score'] + 1 
delivery_df['balls_left'] = 120 - (delivery_df['over'] * 6 + delivery_df['ball'])

# Wickets left using the explicit 'is_wicket' column
delivery_df['is_wicket'] = delivery_df['is_wicket'].fillna(0).astype(int)
delivery_df['wickets_left'] = 10 - delivery_df.groupby('match_id')['is_wicket'].cumsum()

# 6. Calculate Run Rates
delivery_df['crr'] = (delivery_df['current_score'] * 6) / (120 - delivery_df['balls_left'])
delivery_df['rrr'] = (delivery_df['runs_left'] * 6) / delivery_df['balls_left']

# 7. Define the target variable (1 = Win, 0 = Loss)
def result(row):
    return 1 if row['batting_team'] == row['winner'] else 0

delivery_df['result'] = delivery_df.apply(result, axis=1)

# Final Dataset for ML
final_df = delivery_df[['batting_team', 'bowling_team', 'city', 'runs_left', 
                        'balls_left', 'wickets_left', 'total_runs_x', 'crr', 'rrr', 'result']]

# Clean any infinity values or NaNs created by divide-by-zero
final_df = final_df.replace([np.inf, -np.inf], np.nan).dropna().copy()

In [20]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
import pickle

# Split features and target
X = final_df.drop('result', axis=1)
y = final_df['result']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# One-hot encode string columns (teams and cities)
trf = ColumnTransformer([
    ('trf', OneHotEncoder(sparse_output=False, drop='first'), ['batting_team', 'bowling_team', 'city'])
], remainder='passthrough')

# Build the Random Forest Pipeline
pipe = Pipeline(steps=[
    ('step1', trf),
    ('step2', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Train the model
pipe.fit(X_train, y_train)

# Evaluate
y_pred = pipe.predict(X_test)
print(f"Model Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")

# Save the pipeline for the Streamlit dashboard
pickle.dump(pipe, open('pipe.pkl', 'wb'))
print("Success! pipe.pkl has been created.")

Model Accuracy: 99.83%
Success! pipe.pkl has been created.
